In [ ]:
import sys
import importlib
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from matplotlib.ticker import FixedLocator, FuncFormatter, LogLocator, NullFormatter

PYTHON_SRC = Path.cwd() / "python"
if str(PYTHON_SRC) not in sys.path:
    sys.path.insert(0, str(PYTHON_SRC))

import benchmark_reporting.constants as constants
import benchmark_reporting.io as reporting_io
import benchmark_reporting.transforms as transforms
import benchmark_reporting.plotting as plotting

importlib.reload(constants)
importlib.reload(reporting_io)
importlib.reload(transforms)
importlib.reload(plotting)


from benchmark_reporting.constants import *
from benchmark_reporting.io import *
from benchmark_reporting.plotting import *
from benchmark_reporting.transforms import *
from benchmark_reporting.configuration_rectangles import *

BENCHMARK_SUMMARY_JSON = Path("datasets/benchmark_summary.json")
REFERENCE_OVERRIDES = {"hdbscan": "sklearn_brute"}

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.autolayout": True, "figure.dpi": 120})

pd.set_option("display.max_columns", 50)

In [ ]:
df_bench = load_benchmark_data(
    BENCHMARK_SUMMARY_JSON,
    time_field="time_s",
    statistic="median",
)

df_speedup = load_speedup_summary(
    BENCHMARK_SUMMARY_JSON,
    time_field="time_per_algorithm_iteration_s",
    ratio_statistic="median_ratio",
)

TARGET_DATASET = DEFAULT_DATASET


def _filter_target_dataset(df, *, dataset=TARGET_DATASET):
    """Keep the notebook scoped to one benchmark dataset."""
    if df.empty or COL_DATASET not in df.columns:
        return df

    result = df[df[COL_DATASET].astype(str) == str(dataset)].copy()

    for col in result.select_dtypes(include="category").columns:
        result[col] = result[col].cat.remove_unused_categories()

    return result.reset_index(drop=True)


df_bench = filter_primary_references(
    _filter_target_dataset(df_bench),
    overrides=REFERENCE_OVERRIDES,
)
df_speedup = filter_primary_references(
    _filter_target_dataset(df_speedup),
    overrides=REFERENCE_OVERRIDES,
)

print(f"Dataset filter: {TARGET_DATASET!r} only.")
print(f"Reference overrides: {REFERENCE_OVERRIDES}")
print(f"Loaded {len(df_bench)} summarized timing rows.")
print(f"Loaded {len(df_speedup)} post-processed speedup rows.")

display(df_bench.head())
display(df_speedup.head())


In [ ]:
df_exclusions = _filter_target_dataset(load_exclusion_summary(BENCHMARK_SUMMARY_JSON))

if df_exclusions.empty:
    display(Markdown(
        "### Benchmark exclusions\n\n"
        "No benchmark phase/configuration exclusions are recorded in this summary."
    ))
else:
    excluded_phase_configs = len(df_exclusions)

    lines = [
        "### Benchmark exclusions",
        "",
        f"This benchmark summary excludes **{excluded_phase_configs} phase/configuration combination(s)**.\\",
        (
            "These exclusions are intentional and prevent pathological, resource-impractical, or statistically "
            "ill-conditioned cases from distorting the reported timings and speedups."
        ),
        "",
    ]

    grouped = (
        df_exclusions
        .sort_values([COL_PHASE, COL_EXCLUSION_REASON, COL_DIMENSIONS, COL_SAMPLES, COL_CLUSTERS])
        .groupby([COL_PHASE, COL_EXCLUSION_REASON], observed=True)
    )

    for (phase, reason), df_reason in grouped:
        rectangles = summarize_configuration_rectangles(
            df_reason,
            config_cols=NUMERIC_CONFIG_COLS,
        )
        combo_text = ", ".join(
            f"`{rectangle}`"
            for rectangle in rectangles[COL_CONFIGURATION_RECTANGLE]
        )

        lines.extend([
            f"- **{str(phase)}** — {len(df_reason)} excluded combination(s): {combo_text}.\\",
            f"{str(reason)}",
            "",
        ])

    display(Markdown("\n".join(lines)))

In [ ]:
compile_summary = reporting_io.load_benchmark_summary(BENCHMARK_SUMMARY_JSON).get(
    "compile_artifacts",
    {"enabled": False, "architectures": []},
)
df_compile_artifacts = load_compile_artifact_summary(BENCHMARK_SUMMARY_JSON)

if not compile_summary.get("enabled"):
    display(Markdown(
        "### C++ compile artifacts\n\n"
        "No C++ compile artifacts are recorded in this benchmark summary."
    ))
else:
    architectures = compile_summary.get("architectures", [])
    architecture_text = ", ".join(f"`{architecture}`" for architecture in architectures)

    display(Markdown(
        "### C++ compile artifacts\n\n"
        f"**Compiled architecture(s):** {architecture_text}  \n"
        f"**Executable-size records:** {len(df_compile_artifacts)}"
    ))

    fig = plot_executable_sizes(df_compile_artifacts)
    if fig is not None:
        display(fig)
        plt.close(fig)



In [ ]:
cachegrind_summary = reporting_io.load_benchmark_summary(BENCHMARK_SUMMARY_JSON).get(
    "cachegrind",
    {"enabled": False, "records": []},
)
df_cachegrind = _filter_target_dataset(load_cachegrind_summary(BENCHMARK_SUMMARY_JSON))

if not cachegrind_summary.get("enabled"):
    display(Markdown(
        "### Cachegrind\n\n"
        "Cachegrind was not enabled for this benchmark summary."
    ))
elif df_cachegrind.empty:
    display(Markdown(
        "### Cachegrind\n\n"
        f"Cachegrind was enabled, but no Cachegrind records were found for dataset `{TARGET_DATASET}`."
    ))
else:
    cache_model = cachegrind_summary.get("cache_model", {})
    display(Markdown(
        "### Cachegrind\n\n"
        "The measured region is one `bench_case.run_once()` with setup/input loading excluded.  \n"
        "Plots show only the `full` stage. Each metric has a `D` sweep at the median observed `N/K` values and an `N` sweep at the median observed `D/K` values.  \n"
        "For an even number of observed values, the lower middle value is used.  \n"
        f"**Records:** {len(df_cachegrind)}  \n"
        f"**Cache model:** I1=`{cache_model.get('I1')}`, "
        f"D1=`{cache_model.get('D1')}`, LL=`{cache_model.get('LL')}`"
    ))

    # Default: keep the report focused on the two primary read-miss counters:
    # D1mr and DLmr. To inspect write misses, total data misses, references,
    # and miss-rate plots, pass CACHEGRIND_EXTENDED_REPORT_PLOTS below.
    cachegrind_plot_specs = CACHEGRIND_REPORT_PLOTS
    # cachegrind_plot_specs = CACHEGRIND_EXTENDED_REPORT_PLOTS

    for fig in plot_cachegrind_report(
        df_cachegrind,
        plot_specs=cachegrind_plot_specs,
    ):
        display(fig)
        plt.close(fig)



In [ ]:
spill_summary = reporting_io.load_benchmark_summary(BENCHMARK_SUMMARY_JSON).get(
    "spill_detection",
    {"enabled": False},
)
df_spill_detection = load_spill_detection_summary(BENCHMARK_SUMMARY_JSON)

if not spill_summary.get("enabled"):
    display(Markdown(
        "### Spill detection\n\n"
        "Spill detection was not run for this benchmark summary."
    ))
elif spill_summary.get("status") == "ERROR":
    display(Markdown(
        "### Spill detection\n\n"
        f"**Status:** ERROR  \n"
        f"**Scan targets:** {spill_summary.get('scan_target_count', 0)}  \n"
        f"**Error:** `{spill_summary.get('error', 'unknown error')}`"
    ))
else:
    total_candidate_reload_pairs = int(df_spill_detection["Candidate Reload Pairs"].sum())
    scan_target_count = len(df_spill_detection)

    display(Markdown(
        "### Spill detection\n\n"
        f"**Scan targets:** {scan_target_count}  \n"
        f"**Total candidate reload pairs:** {total_candidate_reload_pairs}"
    ))

    for fig in plot_spill_detection_by_phase(df_spill_detection):
        display(fig)
        plt.close(fig)



In [ ]:
lloyd_tolerance_pct = 1e-6

df_lloyd_parity = filter_primary_references(
    _filter_target_dataset(load_lloyd_parity_summary(summary_json=BENCHMARK_SUMMARY_JSON)),
    overrides=REFERENCE_OVERRIDES,
)

df_gmm_parity = filter_primary_references(
    _filter_target_dataset(load_gmm_parity_summary(summary_json=BENCHMARK_SUMMARY_JSON)),
    overrides=REFERENCE_OVERRIDES,
)

df_hdbscan_parity = filter_primary_references(
    _filter_target_dataset(load_hdbscan_parity_summary(summary_json=BENCHMARK_SUMMARY_JSON)),
    overrides=REFERENCE_OVERRIDES,
)

if df_lloyd_parity.empty:
    print(f"WARNING: No Lloyd parity records found in benchmark_summary.json.")
else:
    lloyd_failures = df_lloyd_parity[df_lloyd_parity["Status"] == "❌ FAIL"]

    if not lloyd_failures.empty:
        print(
            f"WARNING: Found {len(lloyd_failures)} Lloyd configurations outside "
            f"{lloyd_tolerance_pct}% inertia-diff limit!"
        )
        display(lloyd_failures)
    else:
        print(
            f"SUCCESS: All {len(df_lloyd_parity)} Lloyd configurations match "
            f"within {lloyd_tolerance_pct}% inertia-diff limit!"
        )

if df_gmm_parity.empty:
    print(f"WARNING: No GMM parity records found in benchmark_summary.json.")
else:
    gmm_failures = df_gmm_parity[df_gmm_parity["Status"] == "❌ FAIL"]

    if not gmm_failures.empty:
        print(f"WARNING: Found {len(gmm_failures)} GMM parity failures!")
        display(gmm_failures)
    else:
        print(f"SUCCESS: All {len(df_gmm_parity)} GMM configurations pass parity!")

if df_hdbscan_parity.empty:
    print(f"WARNING: No HDBSCAN parity records found for selected reference in benchmark_summary.json.")
else:
    hdbscan_failures = df_hdbscan_parity[df_hdbscan_parity["Status"] == "❌ FAIL"]

    if not hdbscan_failures.empty:
        print(f"WARNING: Found {len(hdbscan_failures)} HDBSCAN parity failures!")
        display(hdbscan_failures)
    else:
        print(f"SUCCESS: All {len(df_hdbscan_parity)} HDBSCAN parity records pass!")



In [ ]:
df_bench = add_time_per_algorithm_iteration_columns(df_bench)


def format_variant_params_title(variant, params):
    if pd.isna(params) or str(params) == "Default":
        return f"Variant: {variant}"
    return f"Variant: {variant}; Params: {params}"


df_lloyd = filter_bench(
    df_bench,
    phase=PHASE_DISPLAY_NAMES["lloyd"],
)

df_gmm = filter_bench(
    df_bench,
    phase=PHASE_DISPLAY_NAMES["gmm"],
)

df_speedup = prepare_speedup_comparison_data(df_speedup)


In [ ]:
plot_fixed_costs_vs_algorithm_iteration_time(
    df_bench,
    algorithm_phase_key="lloyd",
    algorithm_label="Lloyd",
)

plot_fixed_costs_vs_algorithm_iteration_time(
    df_bench,
    algorithm_phase_key="gmm",
    algorithm_label="GMM EM",
    fixed_cost_note=(
        "K-Means++ is center-selection only; GMM weights/precision "
        "initialization is not included."
    ),
    requested_N=300_000,
)
plt.show()


In [ ]:
for phase, df_phase in iter_speedup_phase_data(df_speedup):
    for (variant, params), df_plot in df_phase.groupby([COL_VARIANT, COL_PARAMS], observed=True):
        if df_plot.empty:
            continue

        df_plot = df_plot.copy()
        K_order = sorted(df_plot[COL_CLUSTERS].dropna().unique())

        K_palette = dict(
            zip(
                K_order,
                sns.color_palette("Set1", n_colors=len(K_order)),
            )
        )

        g = sns.relplot(
            data=df_plot,
            x=COL_SAMPLES,
            y=COL_SPEEDUP,
            hue=COL_CLUSTERS,
            hue_order=K_order,
            col=COL_DIMENSIONS,
            col_wrap=3,
            kind="line",
            marker="o",
            palette=K_palette,
            height=4,
            aspect=1.2,
            facet_kws={"sharey": False},
        )

        for dim, ax in g.axes_dict.items():
            subset_dim = df_plot[df_plot[COL_DIMENSIONS] == dim]

            for k, subset in subset_dim.groupby(COL_CLUSTERS, observed=True):
                subset = subset.sort_values(COL_SAMPLES)

                x = subset[COL_SAMPLES].to_numpy()
                y_low = subset[COL_SPEEDUP_CI_LOW].to_numpy()
                y_high = subset[COL_SPEEDUP_CI_HIGH].to_numpy()

                ax.fill_between(
                    x,
                    y_low,
                    y_high,
                    color=K_palette[k],
                    alpha=0.18,
                    linewidth=0,
                )

        style_facet_grid(
            g,
            title=(
                "Median Speedup vs. N with Clustered-Bootstrap CI\n"
                f"Phase: {phase}; {format_variant_params_title(variant, params)}"
            ),
            title_y=1.02,
            x_log=True,
            sample_x_axis=True,
            sample_ticks=df_plot[COL_SAMPLES].dropna().unique(),
            titles_inside=True,
            grid_axis="both",
        )

        for ax in g.axes.flat:
            ymin, ymax = ax.get_ylim()
            ax.set_ylim(bottom=0, top=max(ymax, 1.05))

            ax.axhline(
                y=1,
                color="black",
                linewidth=3,
                linestyle="-",
                alpha=0.85,
                zorder=10,
            )

        g.set_axis_labels("N", "Speedup Multiplier (x)")

        move_facet_legend_to_row_ends(
            g,
            default_title="K",
            anchor=(1, 0.5),
        )

        plt.show()


In [ ]:
import matplotlib.patches as patches
from numpy.typing import NDArray

def add_speedup_ci_hatching(*, ax, subset, heat, heat_raw):
    ci_width_pivot, _ = make_cluster_pivot(
        subset,
        COL_SPEEDUP_ERROR_WIDTH,
        reference_pivot=heat_raw,
    )

    relative_ci_pivot = ci_width_pivot / heat.abs()
    hatch_fraction_df = relative_ci_pivot.clip(lower=0, upper=1)

    hatch_fraction: NDArray[np.float64] = hatch_fraction_df.to_numpy(
        dtype=np.float64,
        na_value=np.nan,
    )

    for row_idx in range(hatch_fraction.shape[0]):
        for col_idx in range(hatch_fraction.shape[1]):
            frac = hatch_fraction[row_idx, col_idx]

            if np.isfinite(frac) and frac > 0:
                ax.add_patch(
                    patches.Rectangle(
                        (col_idx, row_idx),
                        1,
                        float(frac),
                        facecolor=(1, 1, 1, 0.08),
                        edgecolor=(0, 0, 0, 0.55),
                        hatch="////",
                        linewidth=0,
                        zorder=2,
                    )
                )

    for text in ax.texts:
        text.set_zorder(3)


for phase, df_phase in iter_speedup_phase_data(df_speedup):
    
    speedup_vmax = df_phase[COL_SPEEDUP].max()

    for (variant, params), df_variant in df_phase.groupby([COL_VARIANT, COL_PARAMS], observed=True):
        if df_variant.empty:
            continue

        K_values = sorted(df_variant[COL_CLUSTERS].unique())

        legend_handles = [
            patches.Patch(
                facecolor=(1, 1, 1, 0.08),
                edgecolor=(0, 0, 0, 0.55),
                hatch="////",
                label="Hatched area = relative CI width",
            )
        ]

        plot_clustered_heatmap_grid(
            df_variant,
            clusters=K_values,
            value_col=COL_SPEEDUP,
            title=(
                f"Median Speedup Heatmap ({LANG_CPP} vs. {LANG_PY}) by K\n"
                f"Phase: {phase} | Variant: {variant} | Params: {params}"
            ),
            heatmap_kwargs={
                "norm": mcolors.LogNorm(vmin=1, vmax=speedup_vmax),
                "vmin": 1,
                "vmax": speedup_vmax,
            },
            cbar_kws={
                "label": "Median Speedup Multiplier (x)",
                "format": mtick.ScalarFormatter(),
                "ticks": [1, 2, 5, 10, 20, 50, 100],
            },
            fmt=".1f",
            legend_handles=legend_handles,
            post_heatmap=add_speedup_ci_hatching,
        )



In [ ]:
def plot_parity_pressure(df, title, legend_handles=None, annot_col=None):
    if df.empty:
        print(f"Skipping {title}: no data.")
        return

    for (variant, params), df_variant in df.groupby([COL_VARIANT, COL_PARAMS], observed=True):
        if df_variant.empty:
            continue

        df_variant = df_variant.copy()
        K_values = sorted(df_variant[COL_CLUSTERS].unique())
        pressure = pd.to_numeric(df_variant["Parity Pressure"], errors="coerce")

        floor = 1e-5
        vmax = 1.0
        plot_value_col = "Parity Pressure Plot Value"
        df_variant[plot_value_col] = pressure.replace([np.inf, -np.inf], vmax).clip(lower=floor)

        vmin = min(floor, 1.0)
        norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)

        ticks = np.logspace(
            np.floor(np.log10(vmin)),
            np.ceil(np.log10(vmax)),
            int(np.ceil(np.log10(vmax)) - np.floor(np.log10(vmin))) + 1,
        )

        kwargs = {}
        if annot_col is not None:
            kwargs["annot_col"] = annot_col
            kwargs["fmt"] = "" 

        plot_clustered_heatmap_grid(
            df_variant,
            clusters=K_values,
            value_col=plot_value_col,
            title=(
                title
                + f"\nVariant: {variant}; Params: {params}\n"
                + "values < 1 pass; values >= 1 are at or over tolerance"
            ),
            heatmap_kwargs={
                "norm": norm,
            },
            cbar_kws={
                "label": "max(diff / tolerance), log scale",
                "ticks": ticks,
                "format": mtick.FormatStrFormatter("%g"),
            },
            legend_handles=legend_handles,
            **kwargs,
        )


if not df_lloyd_parity.empty:
    df_lloyd_pressure = add_lloyd_parity_pressure(df_lloyd_parity)

    plot_parity_pressure(
        df_lloyd_pressure,
        f"Lloyd Parity Pressure ({LANG_CPP} vs. {LANG_PY})",
    )

if not df_gmm_parity.empty:
    df_gmm_pressure = add_gmm_parity_pressure(df_gmm_parity)
    df_gmm_pressure, pressure_letter_handles, pressure_letter_map = (
        add_initial_letter_annotations(
            df_gmm_pressure,
            source_col="Worst Check",
            annot_col="Worst Check Letter",
        )
    )

    plot_parity_pressure(
        df_gmm_pressure,
        f"GMM Parity Pressure ({LANG_CPP} vs. {LANG_PY})\n"
        "max(lower bound, weights, means, covariances, algorithm iterations VS tolerance); ",
        legend_handles=pressure_letter_handles,
        annot_col="Worst Check Letter",
    )

if not df_hdbscan_parity.empty:
    df_hdbscan_full_pressure = add_hdbscan_full_parity_pressure(df_hdbscan_parity)

    hdbscan_worst_check_label_map = {
        "summary_scalar": "summary scalar",
        "probe_values": "overall probes",
        "label_summary_scalar": "label summary",
        "label_probe_values": "membership-label probes",
        "probability_summary_scalar": "probability summary",
        "probability_probe_values": "value-probability probes",
        "noise_count": "noise count",
        "cluster_count": "cluster count",
    }
    
    df_hdbscan_full_pressure["Worst Check Annotation Label"] = (
        df_hdbscan_full_pressure["Worst Check"]
        .map(hdbscan_worst_check_label_map)
    )

    (
        df_hdbscan_full_pressure,
        hdbscan_pressure_letter_handles,
        hdbscan_pressure_letter_map,
    ) = add_initial_letter_annotations(
        df_hdbscan_full_pressure,
        source_col="Worst Check Annotation Label",
        annot_col="Worst Check Letter",
    )

    plot_parity_pressure(
        df_hdbscan_full_pressure,
        f"HDBSCAN Full-Stage Parity Pressure ({LANG_CPP} vs. {LANG_PY})\n"
        "max(summary, probes, labels, probabilities, noise count, cluster count VS tolerance); ",
        legend_handles=hdbscan_pressure_letter_handles,
        annot_col="Worst Check Letter",
    )

In [ ]:
df_norm = add_speedup_retention(df_speedup)

base_K = df_norm[COL_CLUSTERS].min()
df_norm[COL_PHASE] = df_norm[COL_PHASE].cat.remove_unused_categories()

for phase, df_phase in iter_speedup_phase_data(df_norm):
    for (variant, params), df_variant in df_phase.groupby([COL_VARIANT, COL_PARAMS], observed=True):
        if df_variant.empty:
            continue

        g = sns.relplot(
            data=df_variant,
            x=COL_CLUSTERS,
            y=COL_RETENTION,
            hue=COL_SAMPLES,
            col=COL_DIMENSIONS,
            col_wrap=3,
            kind="line",
            marker="o",
            hue_norm=mcolors.LogNorm(),
            legend="full",
            palette="turbo",
            height=4,
            aspect=1.2,
            facet_kws={"sharey": True},
        )
        
        style_facet_grid(
            g,
            titles_inside=True,
            grid_axis="y",
            integer_x_axis=True,
        )

        g.set_axis_labels("K", "Retention (%)")
        g.set(ylim=(0, None))

        add_facet_suptitle(
            g,
            title=(
                f"PHASE: {phase.upper()}\n"
                f"{format_variant_params_title(variant, params)}\n"
                f"Speedup Retention against {base_K}K\n"
                "Does C++ maintain its speed advantage over Python as K increases?"
            ),
        )

        move_facet_legend_to_row_ends(
            g,
            default_title="N",
            label_formatter=format_abbrev,
            anchor=(1.05, 0.5),
        )

        for ax in g.axes.flat:
            ax.axhline(100, color="red", linestyle="--", alpha=0.5)

        plt.show()


In [ ]:
df_bench = add_time_per_algorithm_iteration_per_sample_columns(
    df_bench,
    statistic="median",
    spread="iqr",
)

median_K = min(
    df_bench[COL_CLUSTERS].unique(),
    key=lambda x: abs(x - df_bench[COL_CLUSTERS].median()),
)

for selected_phase_stage, df_speedup_phase in iter_speedup_phase_data(df_speedup):
    selected_phases = list(df_speedup_phase[COL_PHASE].dropna().astype(str).unique())
    selected_stages = list(df_speedup_phase[COL_STAGE].dropna().astype(str).unique())

    phase_df = filter_bench(
        df_bench,
        clusters=median_K,
        language=LANG_CPP,
        phase=selected_phases,
        stage=selected_stages,
    ).copy()

    if phase_df.empty:
        continue

    for (variant, params), plot_df in phase_df.groupby([COL_VARIANT, COL_PARAMS], observed=True):
        if plot_df.empty:
            continue

        plot_df = plot_df.copy()
        plot_df[COL_PHASE] = plot_df[COL_PHASE].cat.remove_unused_categories()
        plot_df[COL_VARIANT] = plot_df[COL_VARIANT].cat.remove_unused_categories()
        plot_df[COL_PARAMS] = plot_df[COL_PARAMS].cat.remove_unused_categories()
        plot_df = plot_df.sort_values([COL_DIMENSIONS, COL_SAMPLES])

        def draw_spread_band(data, **kwargs):
            data = data.sort_values(COL_SAMPLES)

            ax = plt.gca()
            color = kwargs.get("color", sns.color_palette()[0])

            x = data[COL_SAMPLES].to_numpy()
            y_low = data[COL_TIME_PER_ALGORITHM_ITER_PER_SAMPLE_LOW_MS].to_numpy()
            y_high = data[COL_TIME_PER_ALGORITHM_ITER_PER_SAMPLE_HIGH_MS].to_numpy()

            sns.lineplot(
                data=data,
                x=COL_SAMPLES,
                y=COL_TIME_PER_ALGORITHM_ITER_PER_SAMPLE_MS,
                marker="o",
                errorbar=None,
                ax=ax,
                color=color,
            )

            ax.fill_between(
                x,
                y_low,
                y_high,
                color=color,
                alpha=0.2,
                linewidth=0,
            )

        g = sns.FacetGrid(
            data=plot_df,
            col=COL_DIMENSIONS,
            col_wrap=3,
            sharey=False,
            height=4,
            aspect=1.2,
        )

        g.map_dataframe(draw_spread_band)

        spread_label = plot_df[COL_TIMING_RUN_SPREAD].iloc[0]

        style_facet_grid(
            g,
            x_log=True,
            sample_x_axis=True,
            sample_ticks=plot_df[COL_SAMPLES].dropna().unique(),
        )

        g.set_axis_labels(
            "N",
            f"Time per alg iter per sample (ms)",
        )

        add_facet_suptitle(
            g,
            title=(
                f"{LANG_CPP} median time per {selected_phase_stage} algorithm iteration per sample "
                "vs. N\n"
                f"Variant: {variant}; Params: {params}; error bars show timing-run spread ({spread_label}); "
                f"K = {median_K}"
            ),
        )

        plt.show()

In [ ]:
import matplotlib.ticker as mtick

from benchmark_reporting.lloyd_modeling import (
    COL_MODEL_NAME,
    COL_COEFFICIENT,
    COL_PRED_TIME_PER_ALGORITHM_ITER_MS,
    COL_WORK_SIZE,
    fit_lloyd_timing_models,
)

if df_lloyd.empty:
    display(Markdown(
        "### Lloyd timing model\n\n"
        f"No Lloyd timing rows are available for dataset `{TARGET_DATASET}`."
    ))
else:
    lloyd_model_report = fit_lloyd_timing_models(
        df_lloyd,
        vif_threshold=100.0,
        coefficient_threshold=1e-5,
    )

    df_model_predictions = lloyd_model_report.predictions
    df_model_summary = lloyd_model_report.summary
    df_model_coefficients_all = lloyd_model_report.coefficients_all
    df_model_coefficients = lloyd_model_report.coefficients
    df_lloyd_collinearity_removed = lloyd_model_report.collinearity_removed

    print(df_model_summary[COL_MODEL_NAME][0])
    display(df_model_summary.drop(columns=[COL_MODEL_NAME], errors="ignore"))
    display(df_model_coefficients.drop(columns=[COL_MODEL_NAME], errors="ignore"))


    # --- Visualization ----------------------------------------------------------

    for (variant, params), df_plot in df_model_predictions.groupby([COL_VARIANT, COL_PARAMS], observed=True):
        if df_plot.empty:
            continue

        df_cpp = filter_bench(df_plot, language=LANG_CPP).copy()
        df_py = filter_bench(df_plot, language=LANG_PY).copy()

        fig, ax = plt.subplots(figsize=(12, 7))

        sns.scatterplot(
            data=df_cpp,
            x=COL_WORK_SIZE,
            y=COL_PRED_TIME_PER_ALGORITHM_ITER_MS,
            marker="X",
            s=50,
            color="black",
            alpha=0.3,
            label=f"{LANG_CPP} fitted model",
            ax=ax,
        )

        sns.scatterplot(
            data=df_py,
            x=COL_WORK_SIZE,
            y=COL_PRED_TIME_PER_ALGORITHM_ITER_MS,
            marker="P",
            s=50,
            color="black",
            alpha=0.3,
            label=f"{LANG_PY} fitted model",
            ax=ax,
        )

        sns.scatterplot(
            data=df_plot,
            x=COL_WORK_SIZE,
            y=COL_TIME_PER_ALGORITHM_ITER_MS,
            hue=COL_LANGUAGE,
            alpha=1,
            s=25,
            ax=ax,
        )

        ax.set_xscale("log")
        ax.set_yscale("log")

        ax.xaxis.set_major_formatter(
            mtick.FuncFormatter(lambda x, _: format_abbrev(x))
        )

        ax.set_xlabel("Estimated Lloyd dataset size = D × N")
        ax.set_ylabel("Time per Lloyd algorithm iteration (ms)")
        ax.set_title(
            "Lloyd algorithm iteration timing models by language\n"
            f"{format_variant_params_title(variant, params)}"
        )

        ax.grid(axis="both", linestyle="--", alpha=0.7)
        ax.legend()

        plt.tight_layout()
        plt.show()
